## What is Context?
Context is everything that an AI can "see" when it generates a response. 
Formally, context is the set of all information that an AI has access to at any given time while generating a response.

Example:
1. Conversation history in ChatGPT

Context can be very simple like a chat conversation history or complex like a full-fledged chatbot system with multiple conversations and interactions.

2. A Context can exist in distributed systems. For example if you want to implement a feature lets say two factor authentication for you app, following are some of the context that you can use:
    a. Jira issues (for feature description) 
    b. Database (for Data or schema) 
    c. Slack (for communication)
    d. Google Drive (for guidelines)
    e. ...

`Takeaway: Context is scattered and exists accross the distributed systems`

All of these distributed contexts must be put together and feeded into the model for it to generate a response. `Context assembly is a great toil`. This toil is solved using MCPs.

## What is Model Context Protocol(MCP)?

> "Giving LLMs the power of calling functions"

generate an diagram for this line mentioned above

```
┌─────────────┐     read      ┌─────┐    name=abc.txt       ┌─────────────────────┐
│ Read abc.txt│ ────────────► │ LLM │ ─────────────────►    │ def load_file(name):│
└─────────────┘               └─────┘    call load_file     │     ...             │
                                                            └─────────────────────┘
```

## LLMs powered by tools
LLMs process the users prompt and decides which tool to use and how to use it.

```
                                   ┌─────────────────────┐
                         ┌───────► |  1. Weather API     │
                         │         └─────────────────────┘
                         │
┌───────────────┐    ┌───┴───┐     ┌─────────────────────┐
│ Users prompt  ├───►│  LLM  ├───► |    2. Database      |
└───────────────┘    └───┬───┘     └─────────────────────┘
                         │
                         │         ┌─────────────────────┐
                         └───────► │    3. GitHub        │
                                   └─────────────────────┘
```


## Integration Problem
Integration of external tools with LLM is a complex task that requires a lot of reasoning and planning. For each integration, you will need to handle api calls, authentication and authorization, error handling, and more. This can be time-consuming and prone to errors.

If a company has N number of AI chatbots(chatGPT, perxplexity etc) and M number of tools, then total integratoion to handle = N * M

There are more problems like operations, maintenance. Security fragmentation is a big problem in this case.

## Solution of Integration Problem: MCP

MCP standardizes how AI clients connect to external tools via a common protocol. An MCP client connects to multiple MCP servers, each exposing a different service.

```
                              ┌─────────────────────────────┐
                         ┌───►│   MCP Server (Google Drive) │
                         │    └─────────────────────────────┘
                         │
┌────────────────────┐   │    ┌─────────────────────────────┐
│    MCP Client      ├───┼───►│   MCP Server (GitHub)       │
│  (LLM / AI Agent)  │   │    └─────────────────────────────┘
└────────────────────┘   │
                         │    ┌─────────────────────────────┐
                         └───►│   MCP Server (Slack)        │
                              └─────────────────────────────┘
```

- Each **MCP Server** wraps a service and exposes a standard interface
- The **MCP Client** discovers and calls tools without custom integration code
- Adding a new service = adding a new MCP Server, no client changes needed


## Business case: MCP eco-system

**Value Stream:** Become AI native using MCPs (A × B) and scale

```
  A: MCP Servers                          B: MCP Clients
  (Suppliers / Services)                  (AI Chatbots)

  ┌──────────────────┐                   ┌──────────────────┐
  │   Salesforce     │──┐             ┌──│   ChatGPT        │
  └──────────────────┘  │             │  └──────────────────┘
  ┌──────────────────┐  │    MCP      │  ┌──────────────────┐
  │   GitHub         │──┼─ Protocol ──┼──│   Claude         │
  └──────────────────┘  │             │  └──────────────────┘
  ┌──────────────────┐  │             │  ┌──────────────────┐
  │   Slack          │──┘             └──│   Gemini         │
  └──────────────────┘                   └──────────────────┘

       A MCP Servers  ×  B MCP Clients  =  A×B Integrations
                                           (zero custom code per pair)
```

- **A.** Suppliers and service providers build MCP Servers — each wraps a tool or service behind a standard interface.
- **B.** AI chatbots (ChatGPT, Claude, Gemini, etc.) are becoming MCP Client compliant — they can connect to any MCP Server out of the box.
- **A × B:** Every new server is instantly usable by every client. Every new client gains access to every server. The ecosystem grows multiplicatively with no extra integration work.

## Architecture

MCP introduces three distinct components that work together:

| Component | Role | Examples |
|-----------|------|---------|
| **Host** | The AI application the user interacts with | Claude Code, GitHub Copilot, LangChain |
| **LLM** | The model hosted inside the Host that reasons and generates responses | Gemini, ChatGPT, Anthropic Claude |
| **MCP Server** | A backend service wrapped in the MCP protocol, exposing tools | Google Drive, GitHub, Slack |

### Component Diagram

```
  ┌────────────────────────────────────────────────────────────────────┐
  │                         HOST                                       │
  │             (Claude Code / GitHub Copilot / LangChain)             │
  │                                                                    │
  │   ┌────────────────────────────┐    ┌──────────────────────────┐   │
  │   │           LLM              │◄──►│       MCP Client         │   │
  │   │  (Gemini / GPT / Anthropic)│    │    (tool dispatcher)     │   │
  │   └────────────────────────────┘    └──────────────┬───────────┘   │
  └─────────────────────────────────────────────────── ┼ ──────────────┘
                                                       │ MCP Protocol
                    ┌──────────────────────────────────┼───────────────────────┐
                    |                                  |                       |
                    ▼                                  ▼                       ▼
          ┌──────────────────┐             ┌──────────────────┐     ┌──────────────────┐
          │   MCP Server     │             │   MCP Server     │     │   MCP Server     │
          │  (Google Drive)  │             │    (GitHub)      │     │    (Slack)       │
          └──────────────────┘             └──────────────────┘     └──────────────────┘
```

### Dataflow: How a Request is Handled

```
   User             Host                  LLM              MCP Server
    │                 │                    │                    │
    │──1. prompt─────►│                    │                    │
    │                 │──2. forward───────►│                    │
    │                 │                    │                    │
    │                 │◄──3. "I need       │                    │
    │                 │    tool X to       │                    │
    │                 │    answer this"────│                    │
    │                 │                    │                    │
    │                 │──────────4. call tool X────────────────►│
    │                 │◄─────────5. tool result─────────────────│
    │                 │                    │                    │
    │                 │──6. send result───►│                    │
    │                 │◄──7. final─────────│                    │
    │                 │    response        │                    │
    │◄─8. response────│                    │                    │
    │                 │                    │                    │
```

**Step-by-step:**
1. User sends a prompt to the Host
2. Host forwards the prompt to the LLM
3. LLM recognises it needs external data and responds with a tool call (not a final answer)
4. Host's MCP Client calls the appropriate MCP Server
5. MCP Server returns the tool result to the Host
6. Host sends the tool result back to the LLM as additional context
7. LLM now has enough information and generates the final response
8. Host delivers the response to the user

## MCP Client ↔ Server: 1:1 Relationship

Each MCP Client maintains a **dedicated 1:1 connection** per server. The client manages multiple such connections, but each channel is exclusive to one server — servers are never shared across connections.

```
  ┌───────────────────────────────────────────────────────────────────┐
  │                         MCP Client                                │
  │                                                                   │
  │   ┌─────────────────┐   ┌─────────────────┐   ┌─────────────────┐ │
  │   │  Connection 1   │   │  Connection 2   │   │  Connection 3   │ │
  └───┴────────┬────────┴───┴────────┬────────┴───┴────────┬────────┴─┘
               │                     │                     │
               │ 1 : 1               │ 1 : 1               │ 1 : 1
               │                     │                     │
               ▼                     ▼                     ▼
  ┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
  │     MCP Server      │ │     MCP Server      │ │     MCP Server      │
  │   (Google Drive)    │ │      (GitHub)       │ │      (Slack)        │
  └─────────────────────┘ └─────────────────────┘ └─────────────────────┘
```

- One connection serves exactly one server — no sharing, no multiplexing
- Each server only sees its own connection and is unaware of others
- The MCP Client is the single orchestrator managing all 1:1 channels


> `Analogy:` A mobile phone with multiple sim card for multiple service providers.

### Benefits:
1. Decoupling
2. Parrelelism

## MCP Primitives (offerings by server)
1. **Tools** (dynamic)- Actions the AI can ask the server to perform. Example: `list files`, `get file`, `list of commits`

2. **Resources** (static)- Structured data that the AI can access. Example fetching from a database, reading from a file etc.

3. **Prompts** - Predefined prompt templates for the server to use when performing an action. This helps in shaping the AI's behavior. Human users can provide their own prompts, but the server uses the default prompts. Clients gets the standard prompts from the server and converts users prompt to the server prompt format. This is used for reliability and providing guidance for output.

### Prompt Primitive — Example: GitHub Code Review

> **Scenario:** A GitHub MCP Server exposes a prompt called `review-pull-request`. Instead of the user writing a review prompt, the server provides a ready-made, expert-crafted template.

#### Prompt Definition (on the server)

```
  ┌─────────────────────────────────────────────────────────────┐
  │              GitHub MCP Server — Prompt Registry            │
  │                                                             │
  │   name:        "review-pull-request"                        │
  │   description: "Generate a structured code review for a PR" │
  │   arguments:                                                │
  │     ┌──────────────┬──────────┬───────────────────────┐     │
  │     │  Argument    │ Required │  Description          │     │
  │     ├──────────────┼──────────┼───────────────────────┤     │
  │     │  pr_number   │   yes    │  Pull request number  │     │
  │     │  focus       │   no     │  e.g. "security"      │     │
  │     └──────────────┴──────────┴───────────────────────┘     │
  └─────────────────────────────────────────────────────────────┘
```

#### Dataflow

```
   User              Host             GitHub MCP Server          LLM
    │                  │                     │                    │
    │─"review PR #42"─►│                     │                    │
    │                  │                     │                    │
    │                  │──prompts/list──────►│                    │
    │                  │◄──["review-pull-request", "summarize-diff", ...]
    │                  │                     │                    │
    │                  │──prompts/get────────►                    │
    │                  │  name="review-pull-request"              │
    │                  │  args={ pr_number: 42, focus: "security"}│
    │                  │◄──rendered messages─│                    │
    │                  │                     │                    │
    │                  │────────inject rendered prompt───────────►│
    │                  │◄───────────────────────── LLM response───│
    │◄──review result──│                     │                    │
```

#### What the rendered prompt looks like

```
  prompts/get response:
  ┌──────────────────────────────────────────────────────────────────┐
  │  messages: [                                                     │
  │    {                                                             │
  │      role: "user",                                               │
  │      content: "Please review pull request #42.                   │
  │                Focus on: security issues.                        │
  │                Check for: input validation, auth flaws,          │
  │                injection risks, and secrets in code."            │
  │    }                                                             │
  │  ]                                                               │
  └──────────────────────────────────────────────────────────────────┘
```

#### Why this matters

| Without Prompt Primitive | With Prompt Primitive |
|--------------------------|----------------------|
| User must write a detailed review prompt | Host fetches the template — user just says "review PR #42" |
| Prompt quality varies per client | Server ships expert prompts — consistent across all clients |
| Every client re-implements the same logic | Update the template once on the server; all clients benefit instantly |

> **Takeaway:** Prompt primitives let MCP Servers ship *how to ask* alongside *what to do* — the server is the source of truth for both tools and the prompts that use them.

## MCP Primitives: Standard Ops

**Tools:**
1. tools/list - client asks server - "what tools do you provide?"
2. tools/call - client tells server - "call tool X with args Y"

**Resources:**
1. resources/list - client asks server - "what resources do you have?"
2. resources/read - client asks server - "Get me the resource named 'foo'"
3. resources/subscribe or unsubscribe - client subscribes or unsubscribes to a resource's updates

**Prompts**
1. prompts/list - client asks server - "What prompts are available?"
2. prompts/get - client asks server - "get prompt X"

## MCP Data Layer

The data layer defines **how** data is structured and transmitted between MCP clients and servers — it is the common language all participants in the MCP ecosystem agree to speak.

MCP uses **JSON-RPC 2.0** as its data layer. Every message — whether a request, response, or notification — is a JSON object following this standard.

### Message Types

| Type | Direction | Purpose |
|------|-----------|---------|
| **Request** | Client → Server | Ask the server to do something; expects a response |
| **Response** | Server → Client | Result (or error) for a prior request |
| **Notification** | Either direction | Fire-and-forget event; no response expected |

### Example: Tool Call Request & Response

```json
// Request (Client → Server)
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "list_files",
    "arguments": { "path": "/docs" }
  }
}

// Response (Server → Client)
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [{ "type": "text", "text": "README.md\nindex.md" }]
  }
}
```

> **Takeaway:** The MCP protocol defines *what* can be done (tools, resources, prompts); the data layer (JSON-RPC 2.0) defines *how* those operations are expressed and transmitted.

## Why JSON-RPC 2.0 Was Chosen for the MCP Data Layer (Not REST)

REST and JSON-RPC 2.0 are both valid communication protocols, but they are designed for different interaction models. MCP's design makes JSON-RPC 2.0 the natural fit.

### The Core Difference: Resources vs. Procedures

| | REST | JSON-RPC 2.0 |
|---|---|---|
| **Mental model** | Operate on *nouns* (resources) | Call *verbs* (procedures/methods) |
| **Example** | `GET /files/readme.md` | `{"method": "resources/read", ...}` |
| **Transport** | HTTP only (semantics are tied to HTTP verbs) | Any transport: stdio, WebSocket, HTTP |
| **Bidirectional** | No — request/response only | Yes — server can send notifications |
| **Batching** | Not standard | Built-in |

### Reason 1: MCP Operations Are Procedure Calls, Not CRUD

REST is designed around **resources** — you `GET`, `POST`, `PUT`, or `DELETE` a resource identified by a URL. MCP's operations are **actions**:

```
tools/call          → invoke a tool with arguments
resources/read      → read a resource's content
prompts/get         → retrieve a rendered prompt
sampling/createMessage → ask the LLM to generate text
```

These are method calls with named parameters — not CRUD operations on a resource. Forcing them into REST would mean awkward conventions like `POST /tools/{name}/call`, which is just RPC wearing a REST costume.

### Reason 2: Transport Agnosticism

MCP servers run in different environments and must support multiple transports:

```
  stdin/stdout  ──►  JSON-RPC messages  ──►  local process (e.g. Claude Code plugin)
  WebSocket     ──►  JSON-RPC messages  ──►  remote server
  HTTP POST     ──►  JSON-RPC messages  ──►  web-based service
```

REST is semantically tied to HTTP. JSON-RPC 2.0 is just a **message envelope** — the same format works over any byte stream. This is essential for local MCP servers that communicate over `stdio`, with no HTTP layer at all.

### Reason 3: Native Notification Support

JSON-RPC 2.0 has a built-in concept of **notifications** — messages with no `id` and no expected response:

```json
// Server pushes a progress update mid-operation (no response needed)
{
  "jsonrpc": "2.0",
  "method": "notifications/progress",
  "params": {
    "progressToken": "abc123",
    "progress": 60,
    "total": 100
  }
}
```

MCP uses notifications for:
- Progress updates during long-running tool calls
- Log streaming from server to client
- Resource change events (when a resource is updated externally)

REST has no native equivalent — you would need to bolt on SSE or WebSockets separately, creating a hybrid protocol.

> **Takeaway:** REST was designed for the web's resource-oriented, HTTP-bound model. MCP's communication is procedure-oriented, transport-agnostic, and bidirectional — exactly what JSON-RPC 2.0 was built for.